# Direct API Walkthrough

> Plan note: for free-plan accounts, keep GPU usage at 1 and prefer non-parallel real modes (`lumi_v1_6` or `aws_v1_6`).

> This notebook submits and polls compression jobs with direct HTTP requests using a token from `qas auth token`.

In [ ]:
import json
import os
import subprocess
import time

import requests


def _optional_int(name: str):
    raw = os.getenv(name)
    if raw is None or not raw.strip():
        return None
    try:
        return int(raw)
    except ValueError as exc:
        msg = f"Environment variable {name} must be an integer"
        raise RuntimeError(msg) from exc


def _cli_token() -> str:
    try:
        result = subprocess.run(
            ["qas", "auth", "token"],
            capture_output=True,
            check=True,
            text=True,
            timeout=30,
        )
    except (subprocess.CalledProcessError, OSError, subprocess.TimeoutExpired) as exc:
        msg = "Unable to retrieve token from CLI. Run `qas auth login` first."
        raise RuntimeError(msg) from exc

    token = result.stdout.strip()
    if not token:
        msg = "Empty token received from CLI. Run `qas auth login` again."
        raise RuntimeError(msg)
    return token


def _build_job_payload(circuit: str) -> dict:
    payload = {"circuit": circuit}
    num_gpus = _optional_int("QAS_NUM_GPUS")
    if num_gpus is not None:
        payload["num_gpus"] = num_gpus
    iteration_minutes = _optional_int("QAS_ITERATION_MINUTES")
    if iteration_minutes is not None:
        payload["iteration_time_minutes"] = iteration_minutes
    gate_set = os.getenv("QAS_GATE_SET")
    if gate_set:
        payload["gate_set"] = gate_set
    hpc_mode = os.getenv("QAS_HPC_MODE")
    if hpc_mode:
        payload["hpc_mode"] = hpc_mode
    return payload


def _poll_job(
    base_url: str, job_id: str, headers: dict[str, str], *, poll_interval: int, poll_timeout: int
) -> dict:
    deadline = time.monotonic() + poll_timeout
    job_url = f"{base_url}/api/public/v1/circuit-compression/jobs/{job_id}"
    while True:
        response = requests.get(job_url, headers=headers, timeout=30)
        response.raise_for_status()
        payload = response.json()
        status = payload.get("status")
        if status in {"COMPLETED", "FAILED", "CANCELLED"}:
            return payload
        if time.monotonic() > deadline:
            msg = f"Job {job_id} did not finish within {poll_timeout} seconds"
            raise TimeoutError(msg)
        time.sleep(poll_interval)


def _print_json(title: str, payload: dict):
    print(f"\n=== {title} ===")
    print(json.dumps(payload, indent=2, sort_keys=True))


BASE_URL = os.getenv("QAS_BASE_URL", "https://qas.qmill.com").rstrip("/")
ACCESS_TOKEN = _cli_token()
POLL_INTERVAL = int(os.getenv("QAS_POLL_INTERVAL", "5"))
POLL_TIMEOUT = int(os.getenv("QAS_POLL_TIMEOUT", "300"))

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Accept": "application/json",
    "Content-Type": "application/json",
}

print(f"Configured base URL: {BASE_URL}")
print("Auth source: qas auth token")

In [ ]:
mod5_4_circuit = (
    """
    OPENQASM 2.0;
    include \"qelib1.inc\";
    qreg q[5];
    x q[4];
    rz(pi/2) q[4];
    sx q[4];
    rz(pi/2) q[4];
    cx q[3],q[4];
    rz(-pi/4) q[4];
    cx q[0],q[4];
    rz(pi/4) q[4];
    cx q[3],q[4];
    rz(-pi/4) q[4];
    cx q[0],q[4];
    cx q[0],q[3];
    rz(-pi/4) q[3];
    cx q[0],q[3];
    rz(pi/4) q[0];
    rz(pi/4) q[3];
    rz(pi/4) q[4];
    cx q[3],q[4];
    rz(-pi/4) q[4];
    cx q[2],q[4];
    rz(pi/4) q[4];
    cx q[3],q[4];
    rz(-pi/4) q[4];
    cx q[2],q[4];
    cx q[2],q[3];
    rz(-pi/4) q[3];
    cx q[2],q[3];
    rz(pi/4) q[2];
    rz(pi/4) q[3];
    rz(pi/4) q[4];
    rz(pi/2) q[4];
    sx q[4];
    rz(pi/2) q[4];
    cx q[3],q[4];
    rz(pi/2) q[4];
    sx q[4];
    rz(pi/2) q[4];
    cx q[2],q[4];
    rz(-pi/4) q[4];
    cx q[1],q[4];
    rz(pi/4) q[4];
    cx q[2],q[4];
    rz(-pi/4) q[4];
    cx q[1],q[4];
    cx q[1],q[2];
    rz(-pi/4) q[2];
    cx q[1],q[2];
    rz(pi/4) q[1];
    rz(pi/4) q[2];
    rz(pi/4) q[4];
    rz(pi/2) q[4];
    sx q[4];
    rz(pi/2) q[4];
    cx q[2],q[4];
    rz(pi/2) q[4];
    sx q[4];
    rz(pi/2) q[4];
    cx q[1],q[4];
    rz(-pi/4) q[4];
    cx q[0],q[4];
    rz(pi/4) q[4];
    cx q[1],q[4];
    rz(-pi/4) q[4];
    cx q[0],q[4];
    cx q[0],q[1];
    rz(-pi/4) q[1];
    cx q[0],q[1];
    rz(pi/4) q[0];
    rz(pi/4) q[1];
    rz(pi/4) q[4];
    rz(pi/2) q[4];
    sx q[4];
    rz(pi/2) q[4];
    cx q[1],q[4];
    cx q[0],q[4];
    """
).strip()

job_payload = _build_job_payload(mod5_4_circuit)
_print_json("Submitting Compression Job", job_payload)

submission = requests.post(
    f"{BASE_URL}/api/public/v1/circuit-compression/jobs",
    headers=headers,
    json=job_payload,
    timeout=30,
)
submission.raise_for_status()
job_info = submission.json()
job_id = job_info["job_id"]
_print_json("Submission Response", job_info)
print(f"Tracking job {job_id}...")

In [ ]:
result_payload = _poll_job(
    BASE_URL,
    job_id,
    headers,
    poll_interval=POLL_INTERVAL,
    poll_timeout=POLL_TIMEOUT,
)
_print_json("Final Job Payload", result_payload)
print("Final status:", result_payload.get("status"))

In [ ]:
STOP_JOB = False
if STOP_JOB:
    stop_response = requests.delete(
        f"{BASE_URL}/api/public/v1/circuit-compression/jobs/{job_id}",
        headers={"Authorization": f"Bearer {ACCESS_TOKEN}"},
        timeout=30,
    )
    stop_response.raise_for_status()
    print("Stop requested. Status:", stop_response.json().get("status"))